In [1]:
# !python -m pip install great-expectations
# !pip uninstall -y typing_extensions
# !pip uninstall -y great-expectations
!pip install typing_extensions==4.15.0
!pip install great-expectations[pyspark]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.8.0
    Uninstalling typing_extensions-4.8.0:
      Successfully uninstalled typing_extensions-4.8.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 13.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 28.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 21.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: altair
    Found existing installation: altair 5.1.2
    Uninstalling altair-5.1.2:
      Successfully uninstalled altair-5.1.2


In [3]:
import great_expectations as gx
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType

# Initialize Spark session - needed for working with Spark DataFrames

spark = SparkSession.builder.appName("GreatExpectationsExample").getOrCreate()

# Create sample data with some null values to test the expectation
data = [(25,), (None,), (35,), (-5,)]

# Define the schema for the DataFrame (column name and type)
schema = StructType([StructField("age", IntegerType(), True)])

# Create a Spark DataFrame from the data and schema
df = spark.createDataFrame(data, schema)

# Get the Great Expectations context (main entry point)
context = gx.get_context()

# Add a Spark data source, create an asset for the DataFrame, and get a batch for validation
batch = context.data_sources.add_spark("temp").add_dataframe_asset("temp").add_batch_definition_whole_dataframe("temp").get_batch(batch_parameters={"dataframe": df})

# Create an expectation to check that the 'age' column has no null values
exp = gx.expectations.ExpectColumnValuesToNotBeNull(column="age")

# Run the validation on the batch using the expectation
result = batch.validate(exp)

# Better, more understandable output
print(f"VALIDATION PASSED: {result.success}")
print(f"TOTAL RECORDS CHECKED: {result.result['element_count']}")
print(f"NULL VALUES FOUND: {result.result['unexpected_count']}")
print(f"PERCENTAGE WITH NULLS: {result.result['unexpected_percent']:.1f}%")

# If there are unexpected values (nulls), show some examples
if result.result['unexpected_count'] > 0:
    print(f" PROBLEM ROWS: {result.result.get('partial_unexpected_list', 'N/A')}")


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

VALIDATION PASSED: False
TOTAL RECORDS CHECKED: 4
NULL VALUES FOUND: 1
PERCENTAGE WITH NULLS: 25.0%
 PROBLEM ROWS: [None]


In [4]:
import json

# Create a report dictionary with key validation details

report = {
    "expectation": "No nulls in age",
    "success": result.success,
    "unexpected_count": result.result.get('unexpected_count', 0),
    "total_count": result.result.get('element_count', 0),
    "unexpected_percent": result.result.get('unexpected_percent', 0.0)
}

# Save the report to a JSON file with indentation for readability

with open('validation_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print("Report saved!")

Report saved!
